# Reviewer Revision: Leakage-Controlled Multimodal Coffee Disease Experiments

Core experiments:
1. Sensor-only
2. ViT-only
3. ViT + Sensor
4. ViT + Shuffled Sensor

The checkpoint selection criterion is fixed **before training**: minimum validation cross-entropy loss, with higher validation accuracy as a tie-breaker and the earlier epoch retained if still tied.


In [14]:
from google.colab import files

uploaded = files.upload()

Saving train_efficientnet_b0.py to train_efficientnet_b0.py


In [7]:
from google.colab import files

uploaded = files.upload()

Saving images.zip to images.zip


In [9]:
# Upload project ZIP and images.zip first, then adjust filenames if necessary.
!unzip -q coffee_reviewer_experiments.zip -d /content/
!mkdir -p /content/coffee_images
!unzip -q images.zip -d /content/coffee_images


In [10]:
!pip -q install -r /content/coffee_reviewer_experiments/requirements.txt

In [11]:
# Verify split, source IDs, image hashes, and four-class labels.
!python /content/coffee_reviewer_experiments/src/verify_dataset.py --images-dir /content/coffee_images


{
  "clean_rows": 840,
  "train_original_rows": 672,
  "validation_original_rows": 168,
  "train_augmented_entries": 2016,
  "validation_entries": 168,
  "unique_source_ids": 234,
  "train_unique_sources": 188,
  "validation_unique_sources": 46,
  "source_overlap_count": 0,
  "labels": [
    "Coffee rust",
    "Healthy",
    "Leaf spot",
    "Sooty mold"
  ],
  "exactly_four_expected_labels": true,
  "all_validation_augmentation_id_zero": true,
  "all_train_augmented_split_train": true,
  "missing_images": [],
  "pixel_hash_mismatches": [],
  "status": "PASS"
}


In [12]:
# Capture exact runtime package versions and GPU/CUDA information.
!python /content/coffee_reviewer_experiments/src/capture_environment.py

{
  "python": "3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]",
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.35",
  "cuda_available": true,
  "cuda_version": "12.8",
  "gpu": "Tesla T4",
  "packages": {
    "torch": "2.11.0+cu128",
    "torchvision": "0.26.0+cu128",
    "transformers": "5.15.0",
    "numpy": "2.0.2",
    "pandas": "2.2.3",
    "scikit-learn": "1.6.1",
    "Pillow": "11.3.0"
  }
}


## Experiment 1 — Sensor-only

In [20]:
!python /content/coffee_reviewer_experiments/src/train_experiment.py --experiment sensor_only --images-dir /content/coffee_images

[sensor_only] epoch 01/10 | train loss 1.371182 acc 0.291667 (588/2016) | val loss 1.333453 acc 0.636905 (107/168)
[sensor_only] epoch 02/10 | train loss 1.319468 acc 0.499504 (1007/2016) | val loss 1.273104 acc 0.720238 (121/168)
[sensor_only] epoch 03/10 | train loss 1.257263 acc 0.588790 (1187/2016) | val loss 1.185127 acc 0.726190 (122/168)
[sensor_only] epoch 04/10 | train loss 1.177909 acc 0.647321 (1305/2016) | val loss 1.092226 acc 0.720238 (121/168)
[sensor_only] epoch 05/10 | train loss 1.095779 acc 0.649802 (1310/2016) | val loss 0.986918 acc 0.714286 (120/168)
[sensor_only] epoch 06/10 | train loss 1.000786 acc 0.661210 (1333/2016) | val loss 0.888707 acc 0.720238 (121/168)
[sensor_only] epoch 07/10 | train loss 0.921064 acc 0.672619 (1356/2016) | val loss 0.807380 acc 0.714286 (120/168)
[sensor_only] epoch 08/10 | train loss 0.872959 acc 0.665675 (1342/2016) | val loss 0.752974 acc 0.714286 (120/168)
[sensor_only] epoch 09/10 | train loss 0.843897 acc 0.657242 (1325/2016) 

## Experiment 2 — ViT-only

In [21]:
!python /content/coffee_reviewer_experiments/src/train_experiment.py --experiment vit_only --images-dir /content/coffee_images

preprocessor_config.json: 100% 160/160 [00:00<00:00, 843kB/s]
config.json: 100% 69.7k/69.7k [00:00<00:00, 74.7MB/s]

model.safetensors: downloading bytes:  77% 268M/346M [00:01<00:00, 322MB/s, 21.5MB/s  ]
model.safetensors: reconstructing file:  39% 134M/346M [00:01<00:02, 96.2MB/s]
model.safetensors: downloading bytes:  92% 320M/346M [00:01<00:00, 209MB/s, 28.9MB/s  ]
model.safetensors: downloading bytes: 100% 328M/328M [00:01<00:00, 176MB/s, 30.1MB/s  ]
model.safetensors: reconstructing file: 100% 346M/346M [00:01<00:00, 185MB/s, 32.6MB/s  ]
Loading weights: 100% 198/198 [00:00<00:00, 5559.94it/s]
[transformers] ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.weight   | UNEXPECTED | 
classifier.bias     | UNEXPECTED | 
pooler.dense.bias   | MISSING    | 
pooler.dense.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect id

## Experiment 3 — ViT + Sensor

In [22]:
!python /content/coffee_reviewer_experiments/src/train_experiment.py --experiment multimodal --images-dir /content/coffee_images

Loading weights: 100% 198/198 [00:00<00:00, 7063.22it/s]
[transformers] ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.weight   | UNEXPECTED | 
classifier.bias     | UNEXPECTED | 
pooler.dense.weight | MISSING    | 
pooler.dense.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: Memory Efficient attention defaults to a non-deterministic algorithm. To explicitly enable determinism call torch.use_deterministic_algorithms(True, warn_only=False). (Triggered internally at /pytorch/aten/src/ATen/native/transformers/cuda/attention_backward.cu:900.)
  return Variable._execution_engine.run_backward(  # Calls 

## Experiment 4 — ViT + Shuffled Sensor

In [23]:
!python /content/coffee_reviewer_experiments/src/train_experiment.py --experiment shuffled_sensor --images-dir /content/coffee_images

Loading weights: 100% 198/198 [00:00<00:00, 4630.66it/s]
[transformers] ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.bias     | UNEXPECTED | 
classifier.weight   | UNEXPECTED | 
pooler.dense.bias   | MISSING    | 
pooler.dense.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: Memory Efficient attention defaults to a non-deterministic algorithm. To explicitly enable determinism call torch.use_deterministic_algorithms(True, warn_only=False). (Triggered internally at /pytorch/aten/src/ATen/native/transformers/cuda/attention_backward.cu:900.)
  return Variable._execution_engine.run_backward(  # Calls 

## Supplementary sensor-shortcut diagnostics

In [24]:
!python /content/coffee_reviewer_experiments/src/sensor_diagnostic.py

{'method': 'LogisticRegression sensor-only diagnostic', 'train_original_records': 672, 'validation_original_records': 168, 'accuracy': 0.7440476190476191, 'macro_f1': 0.7634802650363074, 'balanced_accuracy': np.float64(0.7678544494720966)}


## Aggregate reviewer comparison table

In [25]:
!python /content/coffee_reviewer_experiments/src/aggregate_results.py
import pandas as pd
display(pd.read_csv('/content/coffee_reviewer_experiments/results/comparison_all_experiments.csv'))


     experiment  selected_epoch  selected_val_loss  record_correct  record_total  record_accuracy  record_macro_f1  record_balanced_accuracy  record_kappa  record_mcc  source_correct  source_total  source_accuracy  source_macro_f1  source_balanced_accuracy  source_kappa  source_mcc                                                checkpoint_sha256
    sensor_only              10           0.678672             120           168         0.714286         0.729467                  0.744264      0.616438    0.618147              35            46          0.76087          0.79707                  0.846429       0.66334    0.672053 ff805645d0da70a00acee56f217bb1e5fa1e2a12b0611314995399ad2a55c537
       vit_only              10           0.000921             168           168         1.000000         1.000000                  1.000000      1.000000    1.000000              46            46          1.00000          1.00000                  1.000000       1.00000    1.000000 616a84247300e2826c2a7

,experiment,selected_epoch,selected_val_loss,record_correct,record_total,record_accuracy,record_macro_f1,record_balanced_accuracy,record_kappa,record_mcc,source_correct,source_total,source_accuracy,source_macro_f1,source_balanced_accuracy,source_kappa,source_mcc,checkpoint_sha256
0,sensor_only,10,0.678672,120,168,0.714286,0.729467,0.744264,0.616438,0.618147,35,46,0.76087,0.79707,0.846429,0.66334,0.672053,ff805645d0da70a00acee56f217bb1e5fa1e2a12b06113...
1,vit_only,10,0.000921,168,168,1.000000,1.000000,1.000000,1.000000,1.000000,46,46,1.00000,1.00000,1.000000,1.00000,1.000000,616a84247300e2826c2a7ac8b57eaadf745a603c755565...
2,multimodal,9,0.001012,168,168,1.000000,1.000000,1.000000,1.000000,1.000000,46,46,1.00000,1.00000,1.000000,1.00000,1.000000,d3c3d9a814f9950aab6197b36319fab2a04fe1db2c3860...
3,shuffled_sensor,9,0.002184,168,168,1.000000,1.000000,1.000000,1.000000,1.000000,46,46,1.00000,1.00000,1.000000,1.00000,1.000000,63351ab9d5a070a10547bc503f9dc8a71b99842290190e...


## Package the exact outputs for the reviewer/repository

In [27]:
!cd /content && zip -qr coffee_reviewer_experiments_RESULTS.zip coffee_reviewer_experiments/results coffee_reviewer_experiments/checkpoints coffee_reviewer_experiments/environment_runtime.json coffee_reviewer_experiments/DATASET_INTEGRITY_REPORT.json
print("Created: /content/coffee_reviewer_experiments_RESULTS.zip")


Created: /content/coffee_reviewer_experiments_RESULTS.zip


In [28]:
!find /content/coffee_reviewer_experiments/checkpoints -name best.pt -type f -print

/content/coffee_reviewer_experiments/checkpoints/multimodal/best.pt
/content/coffee_reviewer_experiments/checkpoints/vit_only/best.pt
/content/coffee_reviewer_experiments/checkpoints/shuffled_sensor/best.pt
/content/coffee_reviewer_experiments/checkpoints/sensor_only/best.pt


In [29]:
from google.colab import files
uploaded = files.upload()

!cp robustness_evaluation.py \
/content/coffee_reviewer_experiments/src/robustness_evaluation.py

Saving robustness_evaluation.py to robustness_evaluation.py


In [30]:
!find /content/coffee_images -type f | head

/content/coffee_images/images/Ds (44).jpg
/content/coffee_images/images/K (4).jpg
/content/coffee_images/images/K (58).jpg
/content/coffee_images/images/Ej (76).png
/content/coffee_images/images/K (88).jpg
/content/coffee_images/images/B (10).png
/content/coffee_images/images/K (80).png
/content/coffee_images/images/K (131).png
/content/coffee_images/images/Ej (141).png
/content/coffee_images/images/Ds (45).jpg


In [31]:
!python /content/coffee_reviewer_experiments/src/robustness_evaluation.py \
    --images-dir /content/coffee_images

Loading weights: 100% 198/198 [00:00<00:00, 5106.76it/s]
[transformers] ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.bias     | UNEXPECTED | 
classifier.weight   | UNEXPECTED | 
pooler.dense.bias   | MISSING    | 
pooler.dense.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading weights: 100% 198/198 [00:00<00:00, 4961.21it/s]
[transformers] ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.bias     | UNEXPECTED | 
classifier.weight   | UNEXPECTED | 
pooler.dense.bias   | MISSING    | 
pooler.dense.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different t

In [32]:
import pandas as pd

summary = pd.read_csv(
    '/content/coffee_reviewer_experiments/results/robustness/robustness_summary.csv'
)

display(
    summary[
        [
            'condition',
            'experiment',
            'record_correct',
            'record_total',
            'record_accuracy',
            'record_macro_f1',
            'source_correct',
            'source_total',
            'source_accuracy',
            'mean_confidence',
            'record_accuracy_drop_from_clean'
        ]
    ]
)

,condition,experiment,record_correct,record_total,record_accuracy,record_macro_f1,source_correct,source_total,source_accuracy,mean_confidence,record_accuracy_drop_from_clean
0,clean,vit_only,168,168,1.00000,1.000000,46,46,1.000000,0.999082,0.00000
1,clean,multimodal,168,168,1.00000,1.000000,46,46,1.000000,0.998991,0.00000
2,clean,shuffled_sensor,168,168,1.00000,1.000000,46,46,1.000000,0.997853,0.00000
3,low_brightness,vit_only,168,168,1.00000,1.000000,46,46,1.000000,0.998743,0.00000
4,low_brightness,multimodal,168,168,1.00000,1.000000,46,46,1.000000,0.997021,0.00000
5,low_brightness,shuffled_sensor,168,168,1.00000,1.000000,46,46,1.000000,0.997130,0.00000
6,low_contrast,vit_only,168,168,1.00000,1.000000,46,46,1.000000,0.999031,0.00000
7,low_contrast,multimodal,168,168,1.00000,1.000000,46,46,1.000000,0.998462,0.00000
8,low_contrast,shuffled_sensor,168,168,1.00000,1.000000,46,46,1.000000,0.997428,0.00000
9,gaussian_blur,vit_only,168,168,1.00000,1.000000,46,46,1.000000,0.996227,0.00000


In [33]:
tab = summary.pivot(
    index='condition',
    columns='experiment',
    values='record_accuracy'
) * 100

display(tab.round(2))

experiment,multimodal,shuffled_sensor,vit_only
condition,,,
clean,100.00,100.0,100.0
combined_degradation,97.62,100.0,100.0
gaussian_blur,100.00,100.0,100.0
low_brightness,100.00,100.0,100.0
low_contrast,100.00,100.0,100.0
partial_occlusion,100.00,100.0,100.0


In [34]:
!cd /content && zip -qr coffee_robustness_RESULTS.zip \
coffee_reviewer_experiments/results/robustness

In [4]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [13]:
!ls /content/coffee_reviewer_experiments
!find /content/coffee_images -type f | head

checkpoints		       README.md
data			       requirements.txt
DATASET_INTEGRITY_REPORT.json  results
environment_runtime.json       Reviewer_Rerun_Experiments_Colab.ipynb
EXPERIMENT_PROTOCOL.md	       src
/content/coffee_images/images/Ds (44).jpg
/content/coffee_images/images/K (4).jpg
/content/coffee_images/images/K (58).jpg
/content/coffee_images/images/Ej (76).png
/content/coffee_images/images/K (88).jpg
/content/coffee_images/images/B (10).png
/content/coffee_images/images/K (80).png
/content/coffee_images/images/K (131).png
/content/coffee_images/images/Ej (141).png
/content/coffee_images/images/Ds (45).jpg


In [15]:
from google.colab import files
uploaded = files.upload()

!cp train_efficientnet_b0.py \
/content/coffee_reviewer_experiments/src/train_efficientnet_b0.py

Saving train_efficientnet_b0.py to train_efficientnet_b0 (1).py


In [16]:
!python /content/coffee_reviewer_experiments/src/train_efficientnet_b0.py \
    --images-dir /content/coffee_images

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth
100% 20.5M/20.5M [00:00<00:00, 175MB/s]
[efficientnet_b0] epoch 01/10 | train loss 1.039617 acc 0.703869 (1419/2016) | val loss 0.567419 acc 0.964286 (162/168)
[efficientnet_b0] epoch 02/10 | train loss 0.402261 acc 0.940972 (1897/2016) | val loss 0.148132 acc 1.000000 (168/168)
[efficientnet_b0] epoch 03/10 | train loss 0.202889 acc 0.960317 (1936/2016) | val loss 0.057248 acc 1.000000 (168/168)
[efficientnet_b0] epoch 04/10 | train loss 0.131646 acc 0.972222 (1960/2016) | val loss 0.060717 acc 1.000000 (168/168)
[efficientnet_b0] epoch 05/10 | train loss 0.111212 acc 0.976687 (1969/2016) | val loss 0.053365 acc 1.000000 (168/168)
[efficientnet_b0] epoch 06/10 | train loss 0.072815 acc 0.985615 (1987/2016) | val loss 0.025740 acc 1.000000 (168/168)
[efficientnet_b0] epoch 07/10 | train loss 0.069366 acc 0.984127 (1984/20

In [17]:
!cd /content && zip -qr efficientnet_b0_RESULTS.zip \
coffee_reviewer_experiments/results/efficientnet_b0 \
coffee_reviewer_experiments/checkpoints/efficientnet_b0

In [18]:
from google.colab import files
uploaded = files.upload()

Saving train_maxvit_tiny.py to train_maxvit_tiny.py


In [19]:
!cp train_maxvit_tiny.py \
/content/coffee_reviewer_experiments/src/train_maxvit_tiny.py

In [20]:
!python /content/coffee_reviewer_experiments/src/train_maxvit_tiny.py \
--images-dir /content/coffee_images


model.safetensors: downloading bytes:  49% 60.3M/124M [00:01<00:01, 55.5MB/s, 5.33MB/s  ]
model.safetensors: downloading bytes:  64% 79.1M/124M [00:02<00:00, 66.6MB/s, 6.55MB/s  ]
model.safetensors: reconstructing file:  34% 41.8M/124M [00:02<00:04, 19.4MB/s, 3.49MB/s  ]
model.safetensors: downloading bytes:  81% 100M/124M [00:02<00:00, 67.5MB/s, 8.09MB/s  ] 
model.safetensors: downloading bytes:  93% 115M/124M [00:02<00:00, 69.5MB/s, 9.72MB/s  ]
model.safetensors: downloading bytes: 100% 118M/118M [00:02<00:00, 41.6MB/s, 10.4MB/s  ]
model.safetensors: reconstructing file: 100% 124M/124M [00:02<00:00, 43.8MB/s, 11.3MB/s  ]
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: Memory Efficient attention defaults to a non-deterministic algorithm. To explicitly enable determinism call torch.use_deterministic_algorithms(True, warn_only=False). (Triggered internally at /pytorch/aten/src/ATen/native/transformers/cuda/attention_backward.cu:900.)
  return Variable.

In [21]:
import json, pandas as pd

path='/content/coffee_reviewer_experiments/results/maxvit_tiny'

display(pd.read_csv(path+'/epoch_history.csv'))

with open(path+'/metrics.json') as f:
    metrics=json.load(f)

metrics

,epoch,train_loss,train_accuracy,val_loss,val_accuracy,val_correct,val_incorrect,time
0,1,0.501179,0.906746,0.136319,1.000000,168,0,70.929351
1,2,0.043770,0.993056,0.049114,1.000000,168,0,66.234902
2,3,0.016055,0.995536,0.016186,1.000000,168,0,66.523199
3,4,0.015161,0.995536,0.057065,0.952381,160,8,66.221590
4,5,0.006070,1.000000,0.046768,0.976190,164,4,65.158029
5,6,0.003126,1.000000,0.035173,0.976190,164,4,66.618712
6,7,0.002324,1.000000,0.036721,0.976190,164,4,67.845619
7,8,0.001482,1.000000,0.042375,0.976190,164,4,68.286904
8,9,0.013935,0.996032,0.055231,0.976190,164,4,68.045948
9,10,0.002144,1.000000,0.063937,0.976190,164,4,68.751056


{'experiment': 'maxvit_tiny',
 'selected_epoch': 3,
 'selected_val_loss': 0.016186005458058345,
 'selected_val_accuracy': 1.0,
 'record_level': {'n': 168,
  'correct': 168,
  'incorrect': 0,
  'accuracy': 1.0,
  'macro_precision': 1.0,
  'macro_recall': 1.0,
  'macro_f1': 1.0,
  'weighted_precision': 1.0,
  'weighted_recall': 1.0,
  'weighted_f1': 1.0,
  'balanced_accuracy': 1.0,
  'cohen_kappa': 1.0,
  'mcc': 1.0,
  'wilson95_low': 0.9776453316501321,
  'wilson95_high': 1.0000000000000002,
  'confusion_matrix_labels': ['Leaf spot',
   'Coffee rust',
   'Sooty mold',
   'Healthy'],
  'confusion_matrix': [[51, 0, 0, 0],
   [0, 48, 0, 0],
   [0, 0, 39, 0],
   [0, 0, 0, 30]]},
 'source_level': {'n': 46,
  'correct': 46,
  'incorrect': 0,
  'accuracy': 1.0,
  'macro_precision': 1.0,
  'macro_recall': 1.0,
  'macro_f1': 1.0,
  'weighted_precision': 1.0,
  'weighted_recall': 1.0,
  'weighted_f1': 1.0,
  'balanced_accuracy': 1.0,
  'cohen_kappa': 1.0,
  'mcc': 1.0,
  'wilson95_low': 0.9229264

In [22]:
!cd /content && zip -qr maxvit_tiny_RESULTS.zip \
coffee_reviewer_experiments/results/maxvit_tiny \
coffee_reviewer_experiments/checkpoints/maxvit_tiny